In [1]:
from bio_agent.agents import Planner
from bio_agent.training import *

### generate dataset

In [2]:
tools = [
        {
            "name": "gene_lookup",
            "description": "Retrieve information about a gene.",
            "arguments": [
                "gene_name"
            ],
        },
        {
            "name": "search_pubmed",
            "description": "Search PubMed literature.",
            "arguments": [
                "query"
            ],
        },
        {
            "name": "classify_sequence",
            "description": "Predict protein family.",
            "arguments": [
                "sequence"
            ],
        },
]

In [3]:
generate_dataset("planner-0.0.1", 250, tools)

Dataset: planner-0.0.1
Output:  ../../datasets/planner-0.0.1/train.jsonl
Tools:   gene_lookup, search_pubmed, classify_sequence
Found 250 existing samples.
Dataset already complete.


In [4]:
### validate dataset

In [5]:
validate_dataset("../../datasets/planner-0.0.1/train.jsonl")

[]

### finetune model

In [6]:
from peft import (
    LoraConfig,
    TaskType,
    get_peft_model
)
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
from datasets import load_dataset
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from bio_agent.config import device, base_tokenizer as tokenizer, BASE_MODEL_ID

In [7]:
model = AutoModelForCausalLM.from_pretrained(BASE_MODEL_ID)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

In [8]:
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
    ],
)

In [9]:
dataset = load_dataset("json", data_files="../../datasets/planner-0.0.1/train.jsonl")

In [59]:
dataset['train'][0]

{'messages': [{'role': 'user',
   'content': 'What is the role of the p53 gene in cancer prevention?'},
  {'role': 'assistant',
   'content': '{"nodes": [{"id": "gene_info","tool": "gene_lookup","args": {"gene_name": "p53"},"depends_on": [],"status": null,"progress": null}]}'}]}

In [10]:
model_1 = get_peft_model(model, lora_config)

In [11]:
tokenized_dataset = dataset.map(
    tokenize,
    remove_columns=dataset['train'].column_names,
)
tokenized_dataset

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 250
    })
})

In [12]:
loader = DataLoader(
    dataset=tokenized_dataset['train'],
    batch_size=10,
    collate_fn=collate_fn
)

In [14]:
NUM_EPOCHS = 4

optimizer = torch.optim.Adam(params=model.parameters(),lr=3e-4)
loss = torch.nn.CrossEntropyLoss(ignore_index=-100)

model_1.train()

for epoch in range(NUM_EPOCHS):

    total_loss = 0.0

    for batch in loader:
        print("===")
        input_ids = batch["input_ids"].to(model_1.device)
        attention_mask = batch["attention_mask"].to(model_1.device)
        labels = batch["labels"].to(model_1.device)

        outputs = model_1(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels,
        )

        loss = outputs.loss

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    print(
        f"Epoch {epoch+1}: "
        f"{total_loss / len(loader):.4f}"
    )

===
===
===
===
===
===
===
===
===
===
===
===
===
===
===
===
===
===
===
===
===
===
===
===
===
Epoch 1: 0.6822
===
===
===
===
===
===
===
===
===
===
===
===
===
===
===
===
===
===
===
===
===
===
===
===
===
Epoch 2: 0.1671
===
===
===
===
===
===
===
===
===
===
===
===
===
===
===
===
===
===
===
===
===
===
===
===
===
Epoch 3: 0.1213
===
===
===
===
===
===
===
===
===
===
===
===
===
===
===
===
===
===
===
===
===
===
===
===
===
Epoch 4: 0.0982


In [15]:
model_1.save_pretrained("../_checkpoints/qwen2.5-0.5b/planner-0.0.1")
tokenizer.save_pretrained("../_checkpoints/qwen2.5-0.5b/planner-0.0.1")

('../_checkpoints/qwen2.5-0.5b/planner-0.0.1/tokenizer_config.json',
 '../_checkpoints/qwen2.5-0.5b/planner-0.0.1/chat_template.jinja',
 '../_checkpoints/qwen2.5-0.5b/planner-0.0.1/tokenizer.json')

### generate plan -> DAG

In [24]:
model_1.eval()
prompt = "User: What cancer is TP53 related to and is there any research paper on that cancer?\nAssistant: "
inputs = tokenizer(
    prompt,
    return_tensors="pt"
)


generated = model.generate(
    **inputs,
    max_new_tokens=128,
)
plan = tokenizer.decode(
    generated[0],
    skip_special_tokens=True
)

print(plan)

[transformers] Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


User: What cancer is TP53 related to and is there any research paper on that cancer?
Assistant: {"nodes":[{"id":"gene_lookup_node_1","tool":"gene_lookup","args":{"gene_name":"TP53"},"depends_on":[],"status":null,"progress":null},{"id":"literature_search_node_2","tool":"search_pubmed","args":{"query":"TP53 related cancer"},"depends_on":[],"status":null,"progress":null}]}


In [49]:
for i in loader:
    a = i['input_ids'][0]
    b = i['labels'][0]
    b = b[b !=-100]
    print(tokenizer.decode(a, skip_special_tokens=True))
    
    print("labels", tokenizer.decode(b, skip_special_tokens=True, ignore_index=-100))
    break

User: What is the role of the p53 gene in cancer prevention?
Assistant: {"nodes": [{"id": "gene_info","tool": "gene_lookup","args": {"gene_name": "p53"},"depends_on": [],"status": null,"progress": null}]}
labels {"nodes": [{"id": "gene_info","tool": "gene_lookup","args": {"gene_name": "p53"},"depends_on": [],"status": null,"progress": null}]}


### execute plans -> outputs

In [26]:
from bio_agent.dispatcher import execute

In [ ]:
### score outputs

In [ ]:
def score():
    pass